# Generate Experiment 1 and 2 Graphs

This notebook grabs the collated results file from storage and generates graphs
comparing model performance on the ICWM task

In [318]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
import shutil
import os
import constants

In [319]:
FIGURES_PATH = constants.ROOT_DIR + "/ICWM/Figures"

### Manually Inspect Model Results

In [320]:
import torch
import shutil

STORAGE_PATH = constants.ROOT_DIR + "/ICWM/results/gemma-3-4b-it/world_size_25/dimension_2/activation_window_50/next_token_prediction_null_rule/results.pth"
if os.path.exists("./results.pth"):
  os.remove("./results.pth")
shutil.copy(
    STORAGE_PATH,
    os.path.join(".", "results.pth"),
)

results = torch.load("./results.pth", weights_only=False)


In [321]:
print(results['answer'][100])
print(results['prompt'][100][-100:])
print(results['response'][100][:15])
print(results['grid'][100])

# Generate Graphs for Next Token Prediction

In [322]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp1_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [323]:
results_df.tail()

In [324]:
ntp_results = results_df[results_df["prompt"] == "next_token_prediction"]
ntp_results = ntp_results[ntp_results["constant_context"] == False]

In [325]:
ntp_results["Setting"] = "Dim: " + ntp_results["dimension"].astype(str) + " World: " + ntp_results["world_size"].astype(str) 

prompt_map = {
    "Dim: 1 World: 16": 1,
    "Dim: 1 World: 25": 2,
    "Dim: 2 World: 16": 3,
    "Dim: 2 World: 25": 4,
    }

ntp_results["Complexity"] = ntp_results["Setting"].map(prompt_map)
ntp_results.head(5)

In [326]:
# Make NTP Prepopulated vs. Not Graph
sns.set_theme(style='darkgrid')
ntp_results["Prepopulated"] = ntp_results["prepopulated"]
ax = sns.lineplot(data=ntp_results, x="Complexity", y="acc", hue="Prepopulated", marker="o")
ax.set_xticks(ntp_results['Complexity'])

# Set the labels using the values from the other column
ax.set_xticklabels(ntp_results['Setting'])
plt.legend(
    title="Prefilled",
    loc='lower left',           # Position of the *legend's anchor point*
)
# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Topology")
plt.ylim(0, 1.05)
plt.title("Next Token Prediction", fontsize=14)

plt.savefig("./Exp1_NTP_Prepopulated_vs_Not.jpg", format="jpg", bbox_inches="tight", dpi=600)
if os.path.exists(os.path.join(FIGURES_PATH, "Exp1_NTP_Prepopulated_vs_Not.jpg")):
  os.remove(os.path.join(FIGURES_PATH, "Exp1_NTP_Prepopulated_vs_Not.jpg"))
shutil.copy("./Exp1_NTP_Prepopulated_vs_Not.jpg", os.path.join(FIGURES_PATH, "Exp1_NTP_Prepopulated_vs_Not.jpg"))

## For Ourselves: Check Alternative vs. Original Not Prepopulated NTP

In [327]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp1_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [328]:
ntp_results = results_df[results_df["prompt"].str.contains("next_token_prediction")]
ntp_results = ntp_results[ntp_results["prepopulated"] == False]
ntp_results = ntp_results[ntp_results["constant_context"] == False]

In [329]:
ntp_results["Setting"] = "Dim: " + ntp_results["dimension"].astype(str) + " World: " + ntp_results["world_size"].astype(str) 

ntp_results['Prompt Type'] = np.where(
    ntp_results['prompt'].str.contains('alternative', case=False, na=False),
    'Alt',
    'Standard'
)

prompt_map = {
    "Dim: 1 World: 16": 1,
    "Dim: 1 World: 25": 2,
    "Dim: 2 World: 16": 3,
    "Dim: 2 World: 25": 4,
    }

ntp_results["Complexity"] = ntp_results["Setting"].map(prompt_map)
ntp_results.head(5)

In [330]:
# Make NTP Prepopulated vs. Not Graph
sns.set_theme(style='darkgrid')
ax = sns.lineplot(data=ntp_results, x="Complexity", y="acc", hue="Prompt Type", marker="o")
ax.set_xticks(ntp_results['Complexity'])

# Set the labels using the values from the other column
ax.set_xticklabels(ntp_results['Setting'])
plt.legend(
    title="Prepopulated",
    loc='lower left',           # Position of the *legend's anchor point*
)
# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Topology")
plt.ylim(0, 1.05)
plt.title("Next Token Prediction")
plt.savefig("./Exp1_NTP_Alt_vs_Standard.pdf", format="pdf", bbox_inches="tight")
if os.path.exists(os.path.join(FIGURES_PATH, "Exp1_NTP_Alt_vs_Standard.pdf")):
  os.remove(os.path.join(FIGURES_PATH, "Exp1_NTP_Alt_vs_Standard.pdf"))
shutil.copy("./Exp1_NTP_Alt_vs_Standard.pdf", os.path.join(FIGURES_PATH, "Exp1_NTP_Alt_vs_Standard.pdf"))

## Generate Constant Context NTP Graphs

In [331]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp1_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [332]:
ntp_results = results_df[results_df["prompt"] == "next_token_prediction"]
ntp_results = ntp_results[ntp_results["constant_context"] == True]

In [333]:
ntp_results["Setting"] = "Dim: " + ntp_results["dimension"].astype(str) + " World: " + ntp_results["world_size"].astype(str) 

prompt_map = {
    "Dim: 1 World: 16": 1,
    "Dim: 1 World: 25": 2,
    "Dim: 2 World: 16": 3,
    "Dim: 2 World: 25": 4,
    }

ntp_results["Complexity"] = ntp_results["Setting"].map(prompt_map)
ntp_results.head(5)

In [334]:
# Make NTP Prepopulated vs. Not Graph
sns.set_theme(style='darkgrid')
ax = sns.lineplot(data=ntp_results, x="Complexity", y="acc", marker="o")
ax.set_xticks(ntp_results['Complexity'])

# Set the labels using the values from the other column
ax.set_xticklabels(ntp_results['Setting'])

# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Topology")
plt.ylim(0, 1.05)
plt.title("Next Token Prediction: Constant Context")

plt.savefig("./Exp1_NTP_Constant_Context.pdf", format="pdf", bbox_inches="tight")
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp1_NTP_Constant_Context.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp1_NTP_Constant_Context.pdf"))
gfile.Copy("./Exp1_NTP_Constant_Context.pdf", os.path.join(FIGURES_PATH, "Exp1_NTP_Constant_Context.pdf"))

# Generate Graphs for ICWM

In [335]:
def compute_one_step_baseline(grid_shape):
  # Hold out one row because it's not a possible query, as one step
  # rule would push it out of bounds
  index_shape = list(grid_shape)
  index_shape[0] = index_shape[0] - 1
  index_shape = tuple(index_shape)
  indices = list(np.ndindex(index_shape))
  
  chance_accuracies = []
  for idxs in indices:
    num_options = 0
    for idx in idxs:
      if idx - 1 >= 0:
        num_options += 1
      if idx + 1 < grid_shape[0]:
        num_options += 1
    chance_accuracies.append(1/num_options)
  return chance_accuracies

In [359]:
np.mean(compute_one_step_baseline((5,5)))

In [336]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp1_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [337]:
results_df.head()

In [338]:
icwm_results = results_df[results_df["prompt"].str.contains("no_information")]
icwm_results = icwm_results[icwm_results["metalearning"] == False]
icwm_results = icwm_results[icwm_results["constant_context"] == False]
icwm_results = icwm_results[icwm_results["prepopulated"] == True]
icwm_results.head()

In [339]:
rule_map = {
    "one_step_vertical_rule": 1,
    "two_step_vertical_rule": 2,
    "three_step_knight_rule": 3,
    }
xlabel_map = {
    "one_step_vertical_rule": "One Step",
    "two_step_vertical_rule": "Two Step",
    "three_step_knight_rule": "Three Step",
    }
icwm_results["Rules"] = icwm_results["rule"].map(rule_map)
icwm_results["xlabel"] = icwm_results["rule"].map(xlabel_map)
icwm_results["Jittered Rules"] = np.random.normal(icwm_results["Rules"], 0.1)

icwm_results.head()


In [340]:
# Make ICWM Accuracy Graph
icwm_results["Topology"] = "Dim: " + icwm_results["dimension"].astype(str) + " World: " + icwm_results["world_size"].astype(str)
ax = sns.scatterplot(data=icwm_results, x="Jittered Rules", y="acc", hue="Topology", s=80)
ax.set_xticks(icwm_results['Rules'])

# Set the labels using the values from the other column
ax.set_xticklabels(icwm_results['xlabel'])

# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Rule")

plt.ylim(0, 1)
plt.title("In-Context World Modeling", fontsize=16)

plt.savefig("./Exp2_ICWM_Accuracy.pdf", format="pdf", bbox_inches="tight", dpi=600)
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp2_ICWM_Accuracy.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp2_ICWM_Accuracy.pdf"))
gfile.Copy("./Exp2_ICWM_Accuracy.pdf", os.path.join(FIGURES_PATH, "Exp2_ICWM_Accuracy.pdf"))


## Generate Constant Context Graph

In [341]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp1_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [342]:
icwm_results = results_df[results_df["prompt"].str.contains("no_information")]
icwm_results = icwm_results[icwm_results["metalearning"] == False]
icwm_results = icwm_results[icwm_results["constant_context"] == True]
icwm_results = icwm_results[icwm_results["prepopulated"] == True]
icwm_results.head()

In [343]:
rule_map = {
    "one_step_vertical_rule": 1,
    "two_step_vertical_rule": 2,
    "three_step_knight_rule": 3,
    }
xlabel_map = {
    "one_step_vertical_rule": "One Step",
    "two_step_vertical_rule": "Two Step",
    "three_step_knight_rule": "Three Step",
    }
icwm_results["Rules"] = icwm_results["rule"].map(rule_map)
icwm_results["xlabel"] = icwm_results["rule"].map(xlabel_map)
icwm_results["Jittered Rules"] = np.random.normal(icwm_results["Rules"], 0.1)

icwm_results.head()


In [344]:
# Make ICWM Accuracy Graph
icwm_results["Topology"] = "Dim: " + icwm_results["dimension"].astype(str) + " World: " + icwm_results["world_size"].astype(str)
ax = sns.scatterplot(data=icwm_results, x="Jittered Rules", y="acc", hue="Topology", s=100)
ax.set_xticks(icwm_results['Rules'])

# Set the labels using the values from the other column
ax.set_xticklabels(icwm_results['xlabel'])

# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Rule")

plt.ylim(0, 1)
plt.title("Gemma-27b In-Context World Modeling: Constant Context")

plt.savefig("./Exp2_ICWM_Accuracy_Constant_Context.pdf", format="pdf", bbox_inches="tight")
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp2_ICWM_Accuracy_Constant_Context.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp2_ICWM_Accuracy_Constant_Context.pdf"))
gfile.Copy("./Exp2_ICWM_Accuracy_Constant_Context.pdf", os.path.join(FIGURES_PATH, "Exp2_ICWM_Accuracy_Constant_Context.pdf"))


# Generate Explicit Grid Graph

In [345]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp1_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [346]:
grid_results = results_df[results_df["prompt"].str.contains("explicit")]
grid_results = grid_results[grid_results["metalearning"] == False]
grid_results = grid_results[grid_results["constant_context"] == False]
grid_results = grid_results[grid_results["prepopulated"] == True]
grid_results.head()

In [347]:
rule_map = {
    "one_step_vertical_rule": 1,
    "two_step_vertical_rule": 2,
    "three_step_knight_rule": 3,
    }
xlabel_map = {
    "one_step_vertical_rule": "One Step",
    "two_step_vertical_rule": "Two Step",
    "three_step_knight_rule": "Three Step",
    }
grid_results["Rules"] = grid_results["rule"].map(rule_map)
grid_results["xlabel"] = grid_results["rule"].map(xlabel_map)
grid_results["Jittered Rules"] = np.random.normal(grid_results["Rules"], 0.1)

grid_results.head()

In [348]:
# Make Explicit Grid Accuracy Graph
grid_results["Grid Size"] = grid_results["world_size"].astype(str)
ax = sns.scatterplot(data=grid_results, x="Jittered Rules", y="acc", hue="Grid Size", s=100)
ax.set_xticks(grid_results['Rules'])

# Set the labels using the values from the other column
ax.set_xticklabels(grid_results['xlabel'])

# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Rule")

plt.ylim(0, 1)
plt.title("Explicit Grid Results", fontsize=16)

plt.savefig("./Exp2_Explicit_Grid_Accuracy.pdf", format="pdf", bbox_inches="tight", dpi=600)
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp2_Explicit_Grid_Accuracy.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp2_Explicit_Grid_Accuracy.pdf"))
gfile.Copy("./Exp2_Explicit_Grid_Accuracy.pdf", os.path.join(FIGURES_PATH, "Exp2_Explicit_Grid_Accuracy.pdf"))


# Generate Structure Difference Graphs

In [349]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/prepopulated_results_structure/collated_exp1_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv") 


In [350]:
# Make Representation Divergence between structure and rules graph

curr_results = results_df.copy()
curr_results = curr_results[curr_results["layer"].isin([60, 62, 46, 48, 38, 40, 32, 34])]
layer_map = {62: "Last", 48: "Last", 60: "Third to Last", 46: "Third to Last",
             38: "Third to Last", 40: "Last", 32: "Third to Last", 34: "Last"}
curr_results["Layer"] = curr_results["layer"].map(layer_map)

curr_results["Model"] = curr_results["model"].str.replace("gemma-3-4b-it", "Gemma-4B")
curr_results["Model"] = curr_results["Model"].str.replace("gemma-3-12b-it", "Gemma-12B")
curr_results["Model"] = curr_results["Model"].str.replace("gemma-3-27b-it", "Gemma-27B")
curr_results["Model"] = curr_results["Model"].str.replace("olmo-2-13b-it", "OLMo-13B")

model_order = sorted(curr_results["Model"].unique())
hue_order = sorted(curr_results["Layer"].unique())

g = sns.catplot(data=curr_results, x="Model", y="mean_relative_energy", hue="Layer", kind="bar",
                order=model_order, hue_order=hue_order, zorder=2, legend=True, height=4, aspect=1.5
)
# 1. Access the main axes object from the FacetGrid
ax = g.ax

# 2. Get the legend handles and labels from the current axes
handles, labels = ax.get_legend_handles_labels()

# 3. Remove the old legend (which is outside the figure)
#    Note: This step is important because catplot often places the legend outside by default.
g._legend.remove() 

# 4. Create a new legend inside the axes
#    - bbox_to_anchor: [x, y] coordinates where the legend box should be anchored 
#      (0,0 is bottom-left, 1,1 is top-right of the axes).
#    - loc: Defines which corner of the legend box should be placed at the bbox_to_anchor coordinates.

sns.barplot(
    data=curr_results,
    x="Model",
    y="mean_baseline_energy",
    hue="Layer",
    order=model_order,    # MUST match the catplot above
    hue_order=hue_order,  # MUST match the catplot above
    ax=g.ax,              # Draw on the existing catplot axes
    palette="tab10",      # Explicit palette (Blue/Orange usually)
    alpha=0.5,            # Make it lighter/transparent
    errorbar=None,        # Remove error bars for the baseline to keep it clean
    zorder=1              # Force this to be the background layer
)
if g.ax.get_legend():
    g.ax.get_legend().remove()
    
g.ax.legend(
    handles, 
    labels,
    title='Layer', # Use the original title of the hue variable
    loc='lower left', 
)

plt.axhline(1.0, c="red", linestyle="--")
plt.ylabel("Relative Dirichlet Energy")
plt.xticks(rotation=40)
plt.ylim(0.8, 1.4)
plt.title("Representations: Structure vs. Rules", fontsize=16)

plt.savefig("./Exp2_Structure_Difference.pdf", format="pdf", bbox_inches="tight", dpi=600)
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp2_Structure_Difference.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp2_Structure_Difference.pdf"))
gfile.Copy("./Exp2_Structure_Difference.pdf", os.path.join(FIGURES_PATH, "Exp2_Structure_Difference.pdf"))


# Generate Metalearning Graph (Appendix)

In [351]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp1_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [352]:
metalearning_results = results_df[results_df["metalearning"] == True]
metalearning_results = metalearning_results[metalearning_results["prepopulated"] == True]
metalearning_results = metalearning_results[metalearning_results["constant_context"] == False]
print(len(metalearning_results))
metalearning_results.head()

In [353]:
rule_map = {
    "one_step_vertical_rule": 1,
    "two_step_vertical_rule": 2,
    "three_step_knight_rule": 3,
    }
xlabel_map = {
    "one_step_vertical_rule": "One Step",
    "two_step_vertical_rule": "Two Step",
    "three_step_knight_rule": "Three Step",
    }
metalearning_results["Rules"] = metalearning_results["rule"].map(rule_map)
metalearning_results["xlabel"] = metalearning_results["rule"].map(xlabel_map)
metalearning_results["Jittered Rules"] = np.random.normal(metalearning_results["Rules"], 0.1)

metalearning_results.head()

In [354]:
# Make Metalearning Accuracy Graph
metalearning_results["Grid Size"] = metalearning_results["world_size"].astype(str)
ax = sns.lineplot(data=metalearning_results, x="Rules", y="acc", marker="o")
ax.set_xticks(metalearning_results['Rules'])

# Set the labels using the values from the other column
ax.set_xticklabels(metalearning_results['xlabel'])

# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Rule")

plt.ylim(0, 1)
plt.title("Metalearning Results")

plt.savefig("./Exp2_Metalearning_Accuracy.pdf", format="pdf", bbox_inches="tight")
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp2_Metalearning_Accuracy.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp2_Metalearning_Accuracy.pdf"))
gfile.Copy("./Exp2_Metalearning_Accuracy.pdf", os.path.join(FIGURES_PATH, "Exp2_Metalearning_Accuracy.pdf"))
